# 语义内核

在这个代码示例中，您将使用 [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI 框架来创建一个基础代理。

本示例的目标是向您展示在后续代码示例中实现不同代理模式时将使用的步骤。


## 导入所需的 Python 包


In [8]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## 创建客户端

在本示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`ai_model_id` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场中可用的其他模型，以查看不同的结果。

为了使用 `Azure Inference SDK`（用于 GitHub Models 的 `base_url`），我们将在 Semantic Kernel 中使用 `OpenAIChatCompletion` 连接器。此外，还有其他 [可用连接器](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion)，可以将 Semantic Kernel 用于其他模型提供商。


In [9]:
import random   

# -----------------------------------------------------
# 1. 定义一个简单的代理工具 (Plugin)
# -----------------------------------------------------

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""
    """一个包含随机度假目的地的插件/工具。"""

    def __init__(self):
        # List of vacation destinations
        # 存储所有可能的度假目的地列表
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        # 用于跟踪上一个目的地，以避免重复推荐
        self.last_destination = None

    # 使用 @kernel_function 装饰器，将这个方法暴露给 AI 模型作为工具
    # 使用 Annotated 为返回值添加详细的类型和描述，模型会利用这些信息来决定何时调用此函数
    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        print("🎯 DEBUG: get_random_destination() 被调用！") 
        # Get available destinations (excluding last one if possible)
        # 复制目的地列表
        available_destinations = self.destinations.copy()
        # 如果有上一个目的地，且列表不止一个，则从可用列表中移除上一个目的地，确保新的目的地不同
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        # 从过滤后的列表中随机选择一个目的地
        destination = random.choice(available_destinations)

        # Update the last destination
        # 更新上一个目的地
        self.last_destination = destination

        # 返回随机选择的目的地字符串
        return destination

In [10]:
# -----------------------------------------------------
# 2. 初始化配置和连接器
# -----------------------------------------------------
load_dotenv() # 从当前目录加载 .env 文件中的环境变量
# 创建 AsyncOpenAI 客户端实例
# model_name="qwen-max"
# client = AsyncOpenAI(
#     api_key=os.environ.get("DASHSCOPE_API_KEY"), 
#     base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
# )

# 使用GPT大模型，作为客户端
model_name = "gpt-4o-mini"
client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com/"
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
# 创建 AI 服务对象，作为 Semantic Kernel 与 Qwen-Max 模型通信的桥梁
chat_completion_service = OpenAIChatCompletion(
    ai_model_id=model_name,
    async_client=client,
)

## 创建代理

下面我们创建一个名为 `TravelAgent` 的代理。

在这个示例中，我们使用了非常简单的指令。你可以修改这些指令，观察代理会如何做出不同的响应。


In [11]:
# -----------------------------------------------------
# 3. 创建和配置 AI 代理
# ChatCompletionAgent 是 Semantic Kernel (SK) 框架中用于构建**核心 AI 代理（Agent）**的基础类。
# 它的主要职责是定义 AI 的身份、能力和行为，并将这些元素集成起来，以便能够响应用户的请求。
# -----------------------------------------------------
agent = ChatCompletionAgent(
    service=chat_completion_service, # 将聊天服务连接到代理
    plugins=[DestinationsPlugin()], # 将上面定义的工具/插件添加到代理中
    name="TravelAgent", # 为代理命名
    # 为代理设置系统指令 (Instructions)，定义它的角色和能力
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## 运行代理

现在我们可以通过定义 `ChatHistory` 并将 `system_message` 添加到其中来运行代理。我们将使用之前定义的 `AGENT_INSTRUCTIONS`。

在这些定义完成后，我们创建一个 `user_inputs`，它代表用户发送给代理的内容。在这个例子中，我们将消息设置为 `Plan me a sunny vacation`。

你可以随意更改这条消息，看看代理会有怎样不同的回应。


In [7]:
# -----------------------------------------------------
# 4. 异步主程序和执行逻辑
# -----------------------------------------------------
# 注意目前是获取不到

# 定义多轮用户输入
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
	# 初始化对话线程，用于在不同 user_input 之间保持上下文
	thread: ChatHistoryAgentThread | None = None

	# 遍历所有用户输入
	for user_input in user_inputs:
		# 初始化 HTML 输出，显示用户查询
		html_output = (
			f"<div style='margin-bottom:10px'>"
			f"<div style='font-weight:bold'>User:</div>"
			f"<div style='margin-left:20px'>{user_input}</div></div>"
		)

		# 初始化用于捕获流式响应的变量
		agent_name = None
		full_response: list[str] = [] # 存储模型的最终自然语言文本
		function_calls: list[str] = [] # 存储函数调用和结果的日志

		# 用于处理流式 Function Call 参数的临时缓冲区
		current_function_name = None
		argument_buffer = ""

		# ------------------- 核心：流式调用循环 -------------------
		# 调用 invoke_stream 实现流式处理和多步 Function Calling
		async for response in agent.invoke_stream(
			messages=user_input, # 当前的用户消息
			thread=thread, # 传入上一次循环更新的线程，实现多轮对话
		):
			thread = response.thread # **更新线程：** 每次迭代后，将最新的线程状态保存
			agent_name = response.name # 获取代理名称
			content_items = list(response.items) # 将响应项转换为列表进行遍历

			# 遍历流中的所有内容项（关键的流式解析逻辑）
			for item in content_items:
				# print(f"DEBUG: item={item}")
				# 注意： 实际运行结果与代码预期或视频演示不符。目前版本下，该代码无法获取到 Calling function 的信息。
				# 查阅相关源码发现，LLM响应仅在满足特定条件（角色为 ASSISTANT，包含 items 或 usage 元数据，且 items 中不含 FunctionCallContent 或 FunctionResultContent）时才会返回给调用者。
				# 推测原因： 可能是由于包版本更新，导致行为与旧版本或演示内容不一致。
				# ------------------- A. 识别 Function Call (模型决定调用工具) -------------------
				if isinstance(item, FunctionCallContent):
					if item.function_name:
						current_function_name = item.function_name # 记录函数名

					# 累积函数参数：参数 JSON 可能分块流式传输，必须进行拼接
					if isinstance(item.arguments, str):
						argument_buffer += item.arguments

				# ------------------- B. 处理 Function Result (Agent 执行工具后返回的结果) -------------------
				elif isinstance(item, FunctionResultContent):
					# **日志记录调用详情：** 当 FunctionResultContent 返回时，意味着函数执行完成。
					if current_function_name:
						formatted_args = argument_buffer.strip()
						try:
							# 尝试解析并美化 JSON 参数
							parsed_args = json.loads(formatted_args)
							formatted_args = json.dumps(parsed_args)
						except Exception:
							pass # 保留原始字符串，如果不是合法的 JSON

						# 记录完整的函数调用日志
						function_calls.append(f"Calling function: {current_function_name}({formatted_args})")

						# 清理缓冲区，准备下一次可能的函数调用
						current_function_name = None
						argument_buffer = ""

					# 记录函数结果日志
					function_calls.append(f"\nFunction Result:\n\n{item.result}")

				# ------------------- C. 处理 Streaming Text (模型的最终文本回复) -------------------
				elif isinstance(item, StreamingTextContent) and item.text:
					# 累积流式传输的文本块，这些是 Agent 最终说的话
					full_response.append(item.text)


		# ------------------- 5. HTML 渲染 Function Calls 日志 -------------------
		if function_calls:
			html_output += (
				"<div style='margin-bottom:10px'>"
				# 使用 <details> 标签创建可折叠区域，用于隐藏复杂的函数调用日志
				"<details>"
				"<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
				"<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
				"border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
				f"{chr(10).join(function_calls)}" # 连接所有函数调用和结果日志
				"</div></details></div>"
			)

		# ------------------- 6. HTML 渲染最终 AI 回复 -------------------
		html_output += (
			"<div style='margin-bottom:20px'>"
			f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
			# 显示累积的最终文本回复
			f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
		)

		# ------------------- 7. 最终展示 -------------------
		display(HTML(html_output)) # 在 IPython 环境中渲染 HTML

await main()

🎯 DEBUG: get_random_destination() 被调用！
[FUNCTION CALL DETAILS] Plugin: DestinationsPlugin
[FUNCTION CALL DETAILS] Function: get_random_destination
[FUNCTION CALL DETAILS] Call ID: call_eADRAEqoAw4jn5tvuZc9aLPJ
[FUNCTION CALL DETAILS] Arguments: {}
New York, USA


🎯 DEBUG: get_random_destination() 被调用！
[FUNCTION CALL DETAILS] Plugin: DestinationsPlugin
[FUNCTION CALL DETAILS] Function: get_random_destination
[FUNCTION CALL DETAILS] Call ID: call_717pZbOMNOy6ofGXnXAIdrbC
[FUNCTION CALL DETAILS] Arguments: {}
Cairo, Egypt
🎯 DEBUG: get_random_destination() 被调用！
[FUNCTION CALL DETAILS] Plugin: DestinationsPlugin
[FUNCTION CALL DETAILS] Function: get_random_destination
[FUNCTION CALL DETAILS] Call ID: call_PfVLpqyZqMmuKQEjud5WG2Ig
[FUNCTION CALL DETAILS] Arguments: {}
Paris, France



---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
